# 02 — Modélisation : Baselines + Optuna + MLflow

**Objectif :** entraîner et comparer plusieurs modèles de classification (LR, NB, SGD, LinearSVC, XGBoost), optimiser **tous** les modèles avec Optuna, et tracker chaque expérience avec MLflow.

| Étape | Contenu |
|---|---|
| Baselines | LR, NB, SGD, LinearSVC — paramètres par défaut |
| Optimisation | Optuna sur **chaque** modèle (LR, NB, SGD, LinearSVC, XGBoost) |
| Évaluation finale | Meilleur modèle sur le test set (touché une seule fois) |
| Tracking | MLflow — chaque run loggé automatiquement |
| Métrique principale | **F1-macro** (classes déséquilibrées 81/19 %) |

## 0. Imports globaux

### Description de l'étape

Chargement de toutes les librairies nécessaires et configuration du path projet.
MLflow et Optuna sont importés ici mais configurés dans leurs sections dédiées.

In [2]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

import mlflow
import mlflow.sklearn
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.preprocessing import MaxAbsScaler
from sklearn.metrics import (
    f1_score, classification_report,
    ConfusionMatrixDisplay, accuracy_score
)
from xgboost import XGBClassifier
import dill

# ── Path projet ───────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

from settings.params import (
    TARGET_COL, STEMMED_COL, FEATURES_ALL, FEATURES_LINEAR,
    TFIDF_NGRAM_RANGE, TFIDF_MAX_FEATURES, TFIDF_MIN_DF, TFIDF_SUBLINEAR_TF,
    PRIMARY_METRIC, RANDOM_STATE,
)
from src.utils.config import cfg
from src.utils.logger import logger

logger.info("Imports OK")

2026-06-05 20:36:46 | INFO | __main__ | Imports OK


### Analyse & Interprétation

Tous les imports sont chargés. Le path projet est résolu dynamiquement depuis le dossier `notebooks/` — fonctionne quel que soit le poste de travail.

## 1. Chargement des données

### Description de l'étape

Chargement des trois splits produits par le notebook `01-02`. Les fichiers `.parquet` préservent les types exacts (bool, StringDtype, float64). On ne retouche plus les données ici — tout le preprocessing est déjà appliqué.

In [3]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

df_train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
df_val   = pd.read_parquet(PROCESSED_DIR / "val.parquet")
df_test  = pd.read_parquet(PROCESSED_DIR / "test.parquet")

for name, df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    rate = df[TARGET_COL].mean()
    logger.info(f"{name:5} | shape={df.shape} | disaster_rate={rate:.3f}")

df_train.head(3)

2026-06-05 20:46:49 | INFO | __main__ | Train | shape=(7959, 25) | disaster_rate=0.186
2026-06-05 20:46:49 | INFO | __main__ | Val   | shape=(1705, 25) | disaster_rate=0.186
2026-06-05 20:46:49 | INFO | __main__ | Test  | shape=(1706, 25) | disaster_rate=0.186


,id,keyword,location,text,target,label,text_length,word_count,avg_word_length,has_url,...,has_location,clean_text,tokens,token_count,unique_tokens,ttr,preprocessed_text,preprocessed_len,compression_ratio,kw_disaster_rate
0,11300,wreckage,<NA>,"""Barr, a conservative Catholic, blamed spread ...",False,Not Disaster,124.0,20.0,5.25,0,...,0,barr a conservative catholic blamed spread of ...,barr conservative catholic blamed spread secul...,12.0,12.0,1.0,barr conserv cathol blame spread secular moral...,12.0,0.600,0.316
1,922,blazing,"Barling, AR.","""I swear on my dear mothers gravewell, I would...",False,Not Disaster,132.0,24.0,4.54,1,...,1,i swear on my dear mothers gravewell i would d...,swear dear mother gravewell onethat wasnt goin...,9.0,9.0,1.0,swear dear mother gravewel would onethat wasnt...,10.0,0.417,0.000
2,3473,demolished,"Auckland Park, Johannesburg",We saw lots of litter around Ushaka beach (alb...,False,Not Disaster,109.0,16.0,5.88,1,...,1,we saw lots of litter around ushaka beach albe...,saw lot litter around ushaka beach albeit heav...,10.0,10.0,1.0,saw lot litter around ushaka beach albeit heav...,10.0,0.625,0.098


### Analyse & Interprétation

*(à compléter après exécution)*

- **Train** : … lignes — disaster_rate ≈ …
- **Val**   : … lignes — disaster_rate ≈ …
- **Test**  : … lignes — disaster_rate ≈ …

Le déséquilibre (~81/19 %) est présent dans les trois splits grâce à la stratification appliquée dans le notebook `01-02`.

## 2. Construction des matrices de features

### Description de l'étape

Trois matrices construites selon les **recommandations de l'EDA** :

**TF-IDF** (`ngram_range=(1,2)`, `max_features=10 000`, `min_df=3`) :
- Les **bigrams** sont justifiés par l'EDA : tweets courts (~9 tokens), ils capturent des expressions composées très discriminantes (`suicide bomber`, `flash flood`) manquées par les unigrams seuls.
- `min_df=3` filtre efficacement les hapax sans perdre les termes discriminants récurrents.
- `max_features=10 000` couvre les unigrams et bigrams les plus informatifs sans explosion dimensionnelle.

**Features numériques — deux ensembles distincts (recommandation EDA) :**
- `FEATURES_LINEAR` (7 features) → LR, SGD, LinearSVC : `word_count` et `token_count` sont **exclus** car colinéaires avec `text_length` — éviter la multicolinéarité pour les modèles linéaires.
- `FEATURES_ALL` (16 features) → XGBoost uniquement : les arbres gèrent naturellement la redondance par sélection de splits.

**Features toujours incluses** (signal fort validé en EDA) : `kw_disaster_rate` (r=0.44), `has_numbers`, `text_length`, `has_url`, `question_count`.

`MaxAbsScaler` préserve les matrices creuses (sparse). TF-IDF **fitté sur train uniquement** — pas de fuite de données.

In [4]:
# ── Targets ───────────────────────────────────────────────────────────────────
y_train = df_train[TARGET_COL].astype(int).values
y_val   = df_val[TARGET_COL].astype(int).values
y_test  = df_test[TARGET_COL].astype(int).values

# ── TF-IDF (fitté sur train uniquement) ──────────────────────────────────────
tfidf = TfidfVectorizer(
    ngram_range=TFIDF_NGRAM_RANGE,
    max_features=TFIDF_MAX_FEATURES,
    min_df=TFIDF_MIN_DF,
    sublinear_tf=TFIDF_SUBLINEAR_TF,
)
X_train_tfidf = tfidf.fit_transform(df_train[STEMMED_COL])
X_val_tfidf   = tfidf.transform(df_val[STEMMED_COL])
X_test_tfidf  = tfidf.transform(df_test[STEMMED_COL])

# ── Features numériques linéaires — LR / SGD / LinearSVC ─────────────────────
# FEATURES_LINEAR exclut word_count et token_count (colinéaires avec text_length)
scaler = MaxAbsScaler()
X_train_num = scaler.fit_transform(df_train[FEATURES_LINEAR].fillna(0))
X_val_num   = scaler.transform(df_val[FEATURES_LINEAR].fillna(0))
X_test_num  = scaler.transform(df_test[FEATURES_LINEAR].fillna(0))

# ── Matrices combinées : TF-IDF + numériques linéaires ────────────────────────
X_train_comb = hstack([X_train_tfidf, csr_matrix(X_train_num)])
X_val_comb   = hstack([X_val_tfidf,   csr_matrix(X_val_num)])
X_test_comb  = hstack([X_test_tfidf,  csr_matrix(X_test_num)])

# ── Features numériques complètes — XGBoost (toutes les 16 features) ──────────
# FEATURES_ALL conserve word_count, token_count — les arbres gèrent la redondance
X_train_xgb = df_train[FEATURES_ALL].fillna(0).values
X_val_xgb   = df_val[FEATURES_ALL].fillna(0).values
X_test_xgb  = df_test[FEATURES_ALL].fillna(0).values

# ── Poids de déséquilibre pour XGBoost ────────────────────────────────────────
# Ratio majority/minority ≈ 4.38 — calculé sur train uniquement
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())

print(f"TF-IDF seul          : {X_train_tfidf.shape}")
print(f"Combiné (TF-IDF+num) : {X_train_comb.shape}")
print(f"XGBoost (num all)    : {X_train_xgb.shape}")
print(f"scale_pos_weight     : {scale_pos_weight:.3f}")

TF-IDF seul          : (7959, 5590)
Combiné (TF-IDF+num) : (7959, 5597)
XGBoost (num all)    : (7959, 16)
scale_pos_weight     : 4.378


### Analyse & Interprétation

*(à compléter après exécution)*

- **TF-IDF** : … échantillons × 10 000 termes (unigrams + bigrams)
- **Combiné** : … échantillons × (10 000 + 7 features linéaires)
- **XGBoost** : … échantillons × 16 features engineerées
- **scale_pos_weight** : … ≈ 4.38 (ratio Not Disaster / Real Disaster)

**Validation des choix EDA :**
- `FEATURES_LINEAR` = 7 features (sans `word_count`, `token_count`) → multicolinéarité évitée ✓
- `FEATURES_ALL` = 16 features pour XGBoost → redondance tolérée par les arbres ✓
- `ngram_range=(1,2)` → bigrams validés par l'EDA sur tweets courts ✓
- `min_df=3` → hapax filtrés sans perte de signal discriminant ✓

## 3. Configuration MLflow

### Description de l'étape

Initialisation de MLflow. Toutes les runs sont stockées dans `mlflow/` à la racine du projet. Chaque modèle entraîné aura sa propre run avec paramètres, métriques et artefact sauvegardés. Pour visualiser : `make mlflow-ui` depuis la racine.

In [ ]:
MLFLOW_DIR = PROJECT_ROOT / "mlflow"
MLFLOW_DIR.mkdir(exist_ok=True)

mlflow.set_tracking_uri(f"file://{MLFLOW_DIR}")
mlflow.set_experiment(cfg["mlflow"]["experiment_name"])

logger.info(f"Experiment : {cfg['mlflow']['experiment_name']}")
logger.info(f"Tracking   : {mlflow.get_tracking_uri()}")

## 4. Fonctions utilitaires

### Description de l'étape

`train_and_log` centralise l'entraînement, l'évaluation sur val et le logging MLflow. Tous les résultats sont également stockés dans la liste `results` pour la comparaison finale.

In [ ]:
results = []

def train_and_log(model, X_tr, y_tr, X_val, y_val,
                  name: str, params: dict = None, tags: dict = None) -> dict:
    """Entraîne, évalue sur val, logue dans MLflow et retourne les métriques."""
    with mlflow.start_run(run_name=name):
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        metrics = {
            "model":     name,
            "f1_macro":  round(f1_score(y_val, y_pred, average="macro"), 4),
            "f1_class0": round(f1_score(y_val, y_pred, pos_label=0), 4),
            "f1_class1": round(f1_score(y_val, y_pred, pos_label=1), 4),
            "accuracy":  round(accuracy_score(y_val, y_pred), 4),
        }

        mlflow.log_params(params or {})
        mlflow.log_metrics({k: v for k, v in metrics.items() if k != "model"})
        if tags:
            mlflow.set_tags(tags)
        mlflow.sklearn.log_model(model, "model")

    results.append(metrics)
    logger.info(f"{name:25} | F1-macro={metrics['f1_macro']:.4f} | acc={metrics['accuracy']:.4f}")
    return metrics

## 5. Baselines (paramètres par défaut)

### 5.1 Logistic Regression

### Description de l'étape

LR + TF-IDF + features numériques linéaires. `class_weight='balanced'` compense le déséquilibre 81/19 % en pondérant automatiquement la classe minoritaire (Real Disaster) inversement à sa fréquence.

In [ ]:
lr_params = {"C": 1.0, "max_iter": 1000, "solver": "lbfgs"}
lr = LogisticRegression(**lr_params, class_weight="balanced", random_state=RANDOM_STATE)

metrics_lr = train_and_log(
    lr, X_train_comb, y_train, X_val_comb, y_val,
    name="LR_baseline", params=lr_params,
    tags={"phase": "baseline", "features": "tfidf+num_linear"}
)
print(classification_report(y_val, lr.predict(X_val_comb),
                             target_names=["Not Disaster", "Real Disaster"]))

### Analyse & Interprétation

*(à compléter après exécution)*

- F1-macro val : …
- Précision classe 1 (Real Disaster) : … | Rappel classe 1 : …
- Précision classe 0 (Not Disaster)  : … | Rappel classe 0 : …

### 5.2 Naive Bayes

### Description de l'étape

MultinomialNB sur TF-IDF uniquement. NB suppose que toutes les features sont indépendantes (hypothèse naïve) et exige des valeurs ≥ 0. Sert de **plancher de référence** : tout modèle doit faire mieux.

In [ ]:
nb_params = {"alpha": 1.0}
nb = MultinomialNB(**nb_params)

metrics_nb = train_and_log(
    nb, X_train_tfidf, y_train, X_val_tfidf, y_val,
    name="NB_baseline", params=nb_params,
    tags={"phase": "baseline", "features": "tfidf_only"}
)
print(classification_report(y_val, nb.predict(X_val_tfidf),
                             target_names=["Not Disaster", "Real Disaster"]))

### Analyse & Interprétation

*(à compléter après exécution)*

- F1-macro val : …
- Performance attendue : la plus faible des 4 baselines. L'hypothèse d'indépendance des mots est rarement vérifiée en NLP.

### 5.3 SGD Classifier

### Description de l'étape

SGDClassifier avec `loss='modified_huber'` — descente de gradient stochastique. Ce n'est pas un modèle à part : selon le `loss`, il simule une LR (`log_loss`) ou un SVM (`hinge`). `modified_huber` est un compromis robuste entre les deux, adapté au texte.

In [ ]:
sgd_params = {"loss": "modified_huber", "alpha": 1e-4, "max_iter": 1000, "penalty": "l2"}
sgd = SGDClassifier(**sgd_params, class_weight="balanced", random_state=RANDOM_STATE)

metrics_sgd = train_and_log(
    sgd, X_train_comb, y_train, X_val_comb, y_val,
    name="SGD_baseline", params=sgd_params,
    tags={"phase": "baseline", "features": "tfidf+num_linear"}
)
print(classification_report(y_val, sgd.predict(X_val_comb),
                             target_names=["Not Disaster", "Real Disaster"]))

### Analyse & Interprétation

*(à compléter après exécution)*

- F1-macro val : …
- Comparaison avec LR : résultats proches attendus car même famille de modèle.

### 5.4 LinearSVC

### Description de l'étape

SVM linéaire — généralement le plus performant sur du texte TF-IDF. Maximise la marge entre les deux classes. `C` contrôle le compromis biais/variance : petit C = grande marge (plus régularisé).

In [ ]:
svc_params = {"C": 1.0, "max_iter": 2000}
svc = LinearSVC(**svc_params, class_weight="balanced", random_state=RANDOM_STATE)

metrics_svc = train_and_log(
    svc, X_train_comb, y_train, X_val_comb, y_val,
    name="LinearSVC_baseline", params=svc_params,
    tags={"phase": "baseline", "features": "tfidf+num_linear"}
)
print(classification_report(y_val, svc.predict(X_val_comb),
                             target_names=["Not Disaster", "Real Disaster"]))

### Analyse & Interprétation

*(à compléter après exécution)*

- F1-macro val : …
- Comparaison avec LR baseline : gain attendu de … points de F1-macro.

### 5.5 XGBoost

### Description de l'étape

XGBoost baseline avec paramètres par défaut sur les **16 features numériques** (`FEATURES_ALL`). Contrairement aux modèles TF-IDF, XGBoost n'utilise pas le texte brut mais uniquement les features engineerées. `scale_pos_weight` gère le déséquilibre des classes (ratio ≈ 4.38, calculé en section 2).

In [ ]:
xgb_params = {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.3}
xgb = XGBClassifier(
    **xgb_params,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric="logloss",
    verbosity=0,
    random_state=RANDOM_STATE,
)

metrics_xgb = train_and_log(
    xgb, X_train_xgb, y_train, X_val_xgb, y_val,
    name="XGBoost_baseline", params=xgb_params,
    tags={"phase": "baseline", "features": "num_all"}
)
print(classification_report(y_val, xgb.predict(X_val_xgb),
                             target_names=["Not Disaster", "Real Disaster"]))

### Analyse & Interprétation

*(à compléter après exécution)*

- F1-macro val : …
- Comparaison avec modèles TF-IDF : XGBoost utilise des features différentes (numériques uniquement) — complémentaire, pas concurrent.
- `scale_pos_weight` ≈ 4.38 compense le déséquilibre des classes comme `class_weight='balanced'` pour les autres modèles.

## 6. Comparaison des baselines

### Description de l'étape

Tableau récapitulatif et visualisation des 5 baselines (LR, NB, SGD, LinearSVC, XGBoost) avant l'optimisation Optuna. Permet d'identifier les tendances et de mesurer le gain apporté par Optuna sur chaque modèle.

In [ ]:
df_baselines = pd.DataFrame(results).sort_values("f1_macro", ascending=False)
display(df_baselines)

fig = px.bar(
    df_baselines.melt(id_vars="model",
                      value_vars=["f1_macro", "f1_class0", "f1_class1", "accuracy"]),
    x="model", y="value", color="variable", barmode="group",
    title="Comparaison des baselines — Val set",
    labels={"value": "Score", "variable": "Métrique", "model": "Modèle"}
)
fig.update_layout(yaxis_range=[0.4, 1.0])
fig.show()

### Analyse & Interprétation

*(à compléter après exécution)*

**Classement attendu :** NB < SGD < LR ≈ LinearSVC < XGBoost

| Modèle | F1-macro val |
|--------|--------------|
| LR_baseline | … |
| NB_baseline | … |
| SGD_baseline | … |
| LinearSVC_baseline | … |
| XGBoost_baseline | … |

## 7. Optimisation avec Optuna

### Description de l'étape

Optuna explore l'espace des hyperparamètres avec l'algorithme **TPE** (Tree-structured Parzen Estimator) — plus efficace qu'une grille exhaustive. Tous les modèles sont optimisés. Le meilleur modèle de chaque study est réentraîné et loggué dans MLflow.

### 7.1 Logistic Regression + Optuna

### Description de l'étape

Hyperparamètres explorés : `C` (régularisation), `solver`, `max_iter`. 30 trials — suffisant pour LR qui converge rapidement.

In [ ]:
def objective_lr(trial):
    C        = trial.suggest_float("C", 1e-3, 10.0, log=True)
    solver   = trial.suggest_categorical("solver", ["lbfgs", "liblinear"])
    max_iter = trial.suggest_int("max_iter", 500, 2000)

    model = LogisticRegression(C=C, solver=solver, max_iter=max_iter,
                                class_weight="balanced", random_state=RANDOM_STATE)
    model.fit(X_train_comb, y_train)
    return f1_score(y_val, model.predict(X_val_comb), average="macro")

study_lr = optuna.create_study(direction="maximize", study_name="lr_optuna")
study_lr.optimize(objective_lr, n_trials=30, show_progress_bar=True)

print(f"Meilleurs params LR  : {study_lr.best_params}")
print(f"Meilleur F1-macro    : {study_lr.best_value:.4f}")

best_lr = LogisticRegression(**study_lr.best_params,
                              class_weight="balanced", random_state=RANDOM_STATE)
metrics_lr_opt = train_and_log(
    best_lr, X_train_comb, y_train, X_val_comb, y_val,
    name="LR_optuna", params=study_lr.best_params,
    tags={"phase": "optuna", "features": "tfidf+num_linear"}
)
print(classification_report(y_val, best_lr.predict(X_val_comb),
                             target_names=["Not Disaster", "Real Disaster"]))

### Analyse & Interprétation

*(à compléter après exécution)*

- Meilleurs params : C=… | solver=… | max_iter=…
- Gain vs LR baseline : +… points de F1-macro

### 7.2 Naive Bayes + Optuna

### Description de l'étape

Hyperparamètres explorés : `alpha` (lissage de Laplace) et `fit_prior` (utiliser les probabilités a priori des classes). Espace restreint — 30 trials suffisent.

In [ ]:
def objective_nb(trial):
    alpha     = trial.suggest_float("alpha", 1e-3, 10.0, log=True)
    fit_prior = trial.suggest_categorical("fit_prior", [True, False])

    model = MultinomialNB(alpha=alpha, fit_prior=fit_prior)
    model.fit(X_train_tfidf, y_train)
    return f1_score(y_val, model.predict(X_val_tfidf), average="macro")

study_nb = optuna.create_study(direction="maximize", study_name="nb_optuna")
study_nb.optimize(objective_nb, n_trials=30, show_progress_bar=True)

print(f"Meilleurs params NB  : {study_nb.best_params}")
print(f"Meilleur F1-macro    : {study_nb.best_value:.4f}")

best_nb = MultinomialNB(**study_nb.best_params)
metrics_nb_opt = train_and_log(
    best_nb, X_train_tfidf, y_train, X_val_tfidf, y_val,
    name="NB_optuna", params=study_nb.best_params,
    tags={"phase": "optuna", "features": "tfidf_only"}
)
print(classification_report(y_val, best_nb.predict(X_val_tfidf),
                             target_names=["Not Disaster", "Real Disaster"]))

### Analyse & Interprétation

*(à compléter après exécution)*

- Meilleurs params : alpha=… | fit_prior=…
- Gain vs NB baseline : +… points de F1-macro
- Marge d'amélioration limitée — l'espace d'hyperparamètres de NB est petit.

### 7.3 SGD + Optuna

### Description de l'étape

Hyperparamètres explorés : `loss`, `alpha`, `penalty` (L1/L2/elasticnet), `max_iter`. La combinaison `loss` + `penalty` détermine le comportement du modèle (LR-like ou SVM-like).

In [ ]:
def objective_sgd(trial):
    loss     = trial.suggest_categorical("loss", ["modified_huber", "log_loss"])
    alpha    = trial.suggest_float("alpha", 1e-5, 1e-1, log=True)
    penalty  = trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"])
    max_iter = trial.suggest_int("max_iter", 500, 2000)

    model = SGDClassifier(loss=loss, alpha=alpha, penalty=penalty, max_iter=max_iter,
                           class_weight="balanced", random_state=RANDOM_STATE)
    model.fit(X_train_comb, y_train)
    return f1_score(y_val, model.predict(X_val_comb), average="macro")

study_sgd = optuna.create_study(direction="maximize", study_name="sgd_optuna")
study_sgd.optimize(objective_sgd, n_trials=30, show_progress_bar=True)

print(f"Meilleurs params SGD : {study_sgd.best_params}")
print(f"Meilleur F1-macro    : {study_sgd.best_value:.4f}")

best_sgd = SGDClassifier(**study_sgd.best_params,
                          class_weight="balanced", random_state=RANDOM_STATE)
metrics_sgd_opt = train_and_log(
    best_sgd, X_train_comb, y_train, X_val_comb, y_val,
    name="SGD_optuna", params=study_sgd.best_params,
    tags={"phase": "optuna", "features": "tfidf+num_linear"}
)
print(classification_report(y_val, best_sgd.predict(X_val_comb),
                             target_names=["Not Disaster", "Real Disaster"]))

### Analyse & Interprétation

*(à compléter après exécution)*

- Meilleurs params : loss=… | alpha=… | penalty=… | max_iter=…
- Gain vs SGD baseline : +… points de F1-macro

### 7.4 LinearSVC + Optuna

### Description de l'étape

Hyperparamètres explorés : `C` (marge SVM) et `max_iter`. Petit C = grande marge (plus régularisé) | Grand C = petite marge (plus ajusté aux données).

In [ ]:
def objective_svc(trial):
    C        = trial.suggest_float("C", 1e-3, 10.0, log=True)
    max_iter = trial.suggest_int("max_iter", 1000, 5000)

    model = LinearSVC(C=C, max_iter=max_iter,
                       class_weight="balanced", random_state=RANDOM_STATE)
    model.fit(X_train_comb, y_train)
    return f1_score(y_val, model.predict(X_val_comb), average="macro")

study_svc = optuna.create_study(direction="maximize", study_name="svc_optuna")
study_svc.optimize(objective_svc, n_trials=30, show_progress_bar=True)

print(f"Meilleurs params SVC : {study_svc.best_params}")
print(f"Meilleur F1-macro    : {study_svc.best_value:.4f}")

best_svc = LinearSVC(**study_svc.best_params,
                      class_weight="balanced", random_state=RANDOM_STATE)
metrics_svc_opt = train_and_log(
    best_svc, X_train_comb, y_train, X_val_comb, y_val,
    name="LinearSVC_optuna", params=study_svc.best_params,
    tags={"phase": "optuna", "features": "tfidf+num_linear"}
)
print(classification_report(y_val, best_svc.predict(X_val_comb),
                             target_names=["Not Disaster", "Real Disaster"]))

### Analyse & Interprétation

*(à compléter après exécution)*

- Meilleurs params : C=… | max_iter=…
- Gain vs LinearSVC baseline : +… points de F1-macro

### 7.5 XGBoost + Optuna

### Description de l'étape

XGBoost utilise les **16 features numériques engineerées** (`FEATURES_ALL`). `scale_pos_weight` gère le déséquilibre des classes (ratio majority/minority ≈ 4.38). 50 trials pour un espace d'exploration plus large (6 hyperparamètres).

In [ ]:
def objective_xgb(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 500),
        "max_depth":        trial.suggest_int("max_depth", 3, 8),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
    }
    model = XGBClassifier(
        **params,
        scale_pos_weight=scale_pos_weight,
        use_label_encoder=False,
        eval_metric="logloss",
        verbosity=0,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train_xgb, y_train)
    return f1_score(y_val, model.predict(X_val_xgb), average="macro")

study_xgb = optuna.create_study(direction="maximize", study_name="xgb_optuna")
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

print(f"Meilleurs params XGB : {study_xgb.best_params}")
print(f"Meilleur F1-macro    : {study_xgb.best_value:.4f}")

best_xgb = XGBClassifier(
    **study_xgb.best_params,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric="logloss",
    verbosity=0,
    random_state=RANDOM_STATE,
)
metrics_xgb_opt = train_and_log(
    best_xgb, X_train_xgb, y_train, X_val_xgb, y_val,
    name="XGBoost_optuna", params=study_xgb.best_params,
    tags={"phase": "optuna", "features": "num_all"}
)
print(classification_report(y_val, best_xgb.predict(X_val_xgb),
                             target_names=["Not Disaster", "Real Disaster"]))

### Analyse & Interprétation

*(à compléter après exécution)*

- Meilleurs params : n_estimators=… | max_depth=… | learning_rate=… | …
- XGBoost capture des signaux différents des modèles TF-IDF (features numériques) — complémentaire.

## 8. Comparaison finale — Tous les modèles

### Description de l'étape

Comparaison complète des 10 modèles (5 baselines + 5 optimisés Optuna). Le meilleur sur le **val set** (F1-macro) est sélectionné pour l'évaluation finale sur le test set.

In [ ]:
df_all = pd.DataFrame(results).sort_values("f1_macro", ascending=False).reset_index(drop=True)
display(df_all)

fig = px.bar(
    df_all.melt(id_vars="model",
                value_vars=["f1_macro", "f1_class0", "f1_class1", "accuracy"]),
    x="model", y="value", color="variable", barmode="group",
    title="Comparaison finale — Tous les modèles (Val set)",
    labels={"value": "Score", "variable": "Métrique", "model": "Modèle"}
)
fig.update_layout(yaxis_range=[0.4, 1.0], xaxis_tickangle=-35)
fig.show()

best_name = df_all.iloc[0]["model"]
print(f"\nMeilleur modèle : {best_name} — F1-macro={df_all.iloc[0]['f1_macro']:.4f}")

### Analyse & Interprétation

*(à compléter après exécution)*

| Modèle | F1-macro val | Gain vs baseline |
|--------|-------------|------------------|
| LR_baseline | … | — |
| NB_baseline | … | — |
| SGD_baseline | … | — |
| LinearSVC_baseline | … | — |
| XGBoost_baseline | … | — |
| LR_optuna | … | +… |
| NB_optuna | … | +… |
| SGD_optuna | … | +… |
| LinearSVC_optuna | … | +… |
| XGBoost_optuna | … | +… |

**Modèle retenu pour le test set : …**

## 9. Évaluation finale sur le test set

### Description de l'étape

Le test set est touché **une seule fois**, ici. C'est l'évaluation finale qui simule les performances en production. Aucun ajustement de modèle ne doit être fait après cette étape.

In [ ]:
model_registry = {
    "LR_baseline":        (lr,       X_test_comb),
    "NB_baseline":        (nb,       X_test_tfidf),
    "SGD_baseline":       (sgd,      X_test_comb),
    "LinearSVC_baseline": (svc,      X_test_comb),
    "XGBoost_baseline":   (xgb,      X_test_xgb),
    "LR_optuna":          (best_lr,  X_test_comb),
    "NB_optuna":          (best_nb,  X_test_tfidf),
    "SGD_optuna":         (best_sgd, X_test_comb),
    "LinearSVC_optuna":   (best_svc, X_test_comb),
    "XGBoost_optuna":     (best_xgb, X_test_xgb),
}

final_model, X_test_final = model_registry[best_name]
y_pred_test = final_model.predict(X_test_final)

print("=" * 55)
print(f"  Modèle final : {best_name}")
print("=" * 55)
print(classification_report(y_test, y_pred_test,
                             target_names=["Not Disaster", "Real Disaster"]))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_test,
    display_labels=["Not Disaster", "Real Disaster"],
    ax=ax, colorbar=False
)
ax.set_title(f"Matrice de confusion — {best_name} (Test set)")
plt.tight_layout()
plt.show()

f1_test = f1_score(y_test, y_pred_test, average="macro")
logger.info(f"F1-macro TEST : {f1_test:.4f}")

### Analyse & Interprétation

*(à compléter après exécution)*

- **F1-macro test** : …
- **F1-macro val**  : … → écart val/test : … (stable = pas d'overfitting)
- Erreurs principales : … faux positifs | … faux négatifs

## 10. Sauvegarde du meilleur modèle

### Description de l'étape

Sauvegarde du modèle final avec `dill`. L'artefact contient le modèle, le TF-IDF, le scaler et les métadonnées nécessaires à l'API de serving.

In [ ]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

artifact = {
    "model":      final_model,
    "tfidf":      tfidf,
    "scaler":     scaler,
    "model_name": best_name,
    "features":   FEATURES_ALL if "XGBoost" in best_name else FEATURES_LINEAR,
    "f1_val":     df_all.iloc[0]["f1_macro"],
    "f1_test":    f1_test,
}

model_path = MODELS_DIR / f"{best_name}.pkl"
with open(model_path, "wb") as f:
    dill.dump(artifact, f)

logger.info(f"Modèle sauvegardé : {model_path}")
print(f"Modèle sauvegardé : {model_path}")

### Analyse & Interprétation

*(à compléter après exécution)*

- Fichier : `models/….pkl`
- Contenu : modèle + TF-IDF + scaler + liste des features → prêt pour l'API

## 11. Synthèse & Décisions

### Résultats clés

*(à compléter après exécution)*

| Modèle | F1-macro Val | F1-macro Test |
|--------|-------------|---------------|
| LR_baseline | … | — |
| NB_baseline | … | — |
| SGD_baseline | … | — |
| LinearSVC_baseline | … | — |
| XGBoost_baseline | … | — |
| LR_optuna | … | — |
| NB_optuna | … | — |
| SGD_optuna | … | — |
| LinearSVC_optuna | … | — |
| XGBoost_optuna | … | — |
| **Meilleur modèle** | **…** | **…** |

### Décisions

- **Modèle retenu :** …
- **Justification :** meilleur F1-macro sur val (…), stable sur test (…)
- **Observations :** …

### Prochaine étape

**`disastertweets_03_interpretation.ipynb`** — analyse SHAP, erreurs qualitatives, importance des features, tweets mal classifiés.